In [1]:
"""
FinDER (FinanceRAG) retrieval evaluation harness — modular splitter x retriever sweep.
 
Loads FinDER corpus/queries/qrels from LOCAL files (Kaggle export), tries
multiple splitter types x chunk configs x retriever backends, and scores
every combination at multiple k values with Recall@k / NDCG@k.
 
Usage:
    pip install sentence-transformers datasets scikit-learn numpy pandas \
        langchain-text-splitters faiss-cpu chromadb rank_bm25 --break-system-packages
    python finder_splitter_eval.py
 
Not every dependency is required — retrievers/splitters with missing
libraries are skipped automatically (see the try/except ImportError guards).
"""
 
import math
from dataclasses import dataclass
 
import numpy as np
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer


D:\conda_envs\finder\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
D:\conda_envs\finder\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [4]:
#!pip install faiss-cpu chromadb rank-bm25

In [3]:
                    BaseTextSplitter
                           │
            ┌──────────────┼──────────────┐
            │              │              │
      CharSplitter   RecursiveChar   TokenSplitter
            │              │              │
            └──────────────┼──────────────┘
                           │
                   SPLITTER_REGISTRY
                           │
                    SWEEP_CONFIG
                           │
                    Experiment loop

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 3)

In [5]:
# ---------------------------------------------------------------------------
# Local file paths (Kaggle exprt)
# ---------------------------------------------------------------------------
 
CORPUS_PATH = "../finrag_data/finder_corpus.jsonl/corpus.jsonl"
QUERIES_PATH = "../finrag_data/finder_queries.jsonl/queries.jsonl"
QRELS_PATH = "../finrag_data/FinDER_qrels.tsv"
OUT_PATH_SWEEPRESULT = "../results/finder_sweep_results.csv" 

QRELS_COLS = ("query_id", "corpus_id", "score")
 
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # or a local path if huggingface.co is unreachable
K_VALUES = [1, 3, 5, 10, 20]      # evaluate at all these k values in one pass
MAX_K = max(K_VALUES)             # retrieve this many chunks once per query, then slice per k
MAX_QUERIES = 200
MAX_CORPUS_DOCS = None


In [6]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
)


class CharSplitter:
    def __init__(self, chunk_size: int, overlap: int):
        self.chunk_size = chunk_size
        self.overlap = overlap

    def split(self, text: str) -> list[str]:
        if not text:
            return []

        step = self.chunk_size - self.overlap
        chunks = []

        for start in range(0, len(text), step):
            piece = text[start:start + self.chunk_size]

            if piece.strip():
                chunks.append(piece)

            if start + self.chunk_size >= len(text):
                break

        return chunks


class RecursiveCharSplitter:
    def __init__(self, chunk_size: int, overlap: int):
        self.chunk_size = chunk_size
        self.overlap = overlap

        self._splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=overlap,
        )

    def split(self, text: str) -> list[str]:
        return self._splitter.split_text(text)


class TokenSplitter:
    def __init__(self, chunk_size: int, overlap: int):
        self.chunk_size = chunk_size
        self.overlap = overlap

        self._splitter = TokenTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=overlap,
        )

    def split(self, text: str) -> list[str]:
        return self._splitter.split_text(text)



SPLITTER_REGISTRY = {
    "char": CharSplitter,
    "recursive_char": RecursiveCharSplitter,
    "token": TokenSplitter,
}

SWEEP_CONFIG = {
    "char": [(256, 64), (512, 100)],
    "recursive_char": [(256, 64), (512, 100)],
    "token": [(128, 20), (256, 40)],
}


In [7]:

# ---------------------------------------------------------------------------
# Retriever registry — every retriever fn has the same signature:
#   (chunk_texts, chunk_vecs, query_texts, query_vecs, k) -> np.ndarray of shape
#   (n_queries, k) giving chunk indices in ranked order (best first).
# This lets dense (embedding-based) and sparse (keyword-based) retrievers
# plug into the same evaluation loop.
# ---------------------------------------------------------------------------
 
def _cosine_numpy_retriever(chunk_texts, chunk_vecs, query_texts, query_vecs, k):
    sims = query_vecs @ chunk_vecs.T                       # (n_queries, n_chunks); vectors are pre-normalized
    k = min(k, sims.shape[1])
    idx = np.argpartition(-sims, k - 1, axis=1)[:, :k]
    row_scores = np.take_along_axis(sims, idx, axis=1)
    order = np.argsort(-row_scores, axis=1)
    return np.take_along_axis(idx, order, axis=1)
 
 
def _faiss_flat_retriever(chunk_texts, chunk_vecs, query_texts, query_vecs, k):
    import faiss
    dim = chunk_vecs.shape[1]
    print("Shape of Chunk Vec", chunk_vecs.shape[1])
    index = faiss.IndexFlatIP(dim)                          # exact search; inner product == cosine since normalized
    index.add(np.ascontiguousarray(chunk_vecs.astype("float32")))
    _, idx = index.search(np.ascontiguousarray(query_vecs.astype("float32")), k)
    return idx
 
 
def _faiss_hnsw_retriever(chunk_texts, chunk_vecs, query_texts, query_vecs, k):
    import faiss
    dim = chunk_vecs.shape[1]
    index = faiss.IndexHNSWFlat(dim, 32)                    # approximate; much faster at large scale, small recall cost
    index.hnsw.efConstruction = 40
    index.add(np.ascontiguousarray(chunk_vecs.astype("float32")))
    index.hnsw.efSearch = 64
    _, idx = index.search(np.ascontiguousarray(query_vecs.astype("float32")), k)
    return idx
 
 
def _chroma_retriever(chunk_texts, chunk_vecs, query_texts, query_vecs, k):
    import chromadb
    client = chromadb.EphemeralClient()                      # in-memory, no persistence — fresh per eval run
    collection = client.get_or_create_collection(name="finder_eval")
    ids = [str(i) for i in range(len(chunk_texts))]
    # Chroma has an add-size limit per call on some versions; batch to be safe
    batch = 500
    for start in range(0, len(ids), batch):
        collection.add(
            ids=ids[start:start + batch],
            embeddings=chunk_vecs[start:start + batch].tolist(),
            documents=chunk_texts[start:start + batch],
        )
    result = collection.query(query_embeddings=query_vecs.tolist(), n_results=k)
    idx = np.array([[int(i) for i in row] for row in result["ids"]])
    client.delete_collection("finder_eval")              # clear state so the next sweep config starts clean
    return idx
 
 
def _bm25_retriever(chunk_texts, chunk_vecs, query_texts, query_vecs, k):
    from rank_bm25 import BM25Okapi
    tokenized_corpus = [t.lower().split() for t in chunk_texts]
    bm25 = BM25Okapi(tokenized_corpus)
    rows = []
    for q in query_texts:
        scores = bm25.get_scores(q.lower().split())
        top = np.argsort(scores)[::-1][:k]
        rows.append(top)
    return np.array(rows)
 
 
RETRIEVER_REGISTRY = {
    #"cosine_numpy": _cosine_numpy_retriever,
    "faiss_flat": _faiss_flat_retriever,
    "faiss_hnsw": _faiss_hnsw_retriever,
    "chroma": _chroma_retriever,
    "bm25": _bm25_retriever,
}
 
RETRIEVERS_TO_TRY = ["cosine_numpy", "faiss_flat", "chroma", "bm25"]  # comment any out to skip

RETRIEVERS_TO_TRY = ["faiss_flat", "faiss_hnsw", "chroma", "bm25"] 

In [8]:
# ---------------------------------------------------------------------------
# Chunking
# ---------------------------------------------------------------------------
 
@dataclass
class Chunk:
    doc_id: str
    text: str
    

In [9]:

def build_chunks(corpus_rows, splitter_name, chunk_size, overlap):
    splitter_class = SPLITTER_REGISTRY[splitter_name]
    splitter_fn = splitter_class( chunk_size=chunk_size, overlap=overlap)
    chunks = []
    for row in corpus_rows:
        title = (row.get("title") or "").strip()
        body = row.get("text") or ""
        full_text = f"{title}\n{body}" if title else body
        for piece in splitter_fn.split(full_text):
            if piece and piece.strip():
                chunks.append(Chunk(doc_id=str(row["_id"]), text=piece))
    return chunks

In [10]:
# ---------------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------------
 
def dcg(rels):
    return sum(r / math.log2(i + 2) for i, r in enumerate(rels))
 
 
def ndcg_at_k(retrieved_doc_ids, relevant_doc_ids, k):
    rels = [1 if d in relevant_doc_ids else 0 for d in retrieved_doc_ids[:k]]
    ideal = sorted(rels, reverse=True)
    ideal_dcg = dcg(ideal)
    return dcg(rels) / ideal_dcg if ideal_dcg > 0 else 0.0
 
 
def recall_at_k(retrieved_doc_ids, relevant_doc_ids, k):
    if not relevant_doc_ids:
        return 0.0
    hit = len(set(retrieved_doc_ids[:k]) & relevant_doc_ids)
    return hit / len(relevant_doc_ids)
    

In [11]:

# ---------------------------------------------------------------------------
# Data loading
# ---------------------------------------------------------------------------
 
def load_local_data():
    corpus_ds = load_dataset("json", data_files=CORPUS_PATH, split="train")
    queries_ds = load_dataset("json", data_files=QUERIES_PATH, split="train")
    try:
        qrels_ds = load_dataset("csv", data_files=QRELS_PATH, delimiter="\t", split="train")
    except Exception as e:
        raise SystemExit(f"Could not load qrels file '{QRELS_PATH}': {e}")
    return corpus_ds, queries_ds, qrels_ds
 
 
def build_relevant_map(qrels_ds, query_ids):
    qcol, dcol, scol = QRELS_COLS
    relevant = {}
    for row in qrels_ds:
        qid = str(row[qcol])
        if qid not in query_ids:
            continue
        if row.get(scol, 1) and int(row[scol]) > 0:
            relevant.setdefault(qid, set()).add(str(row[dcol]))
    return relevant
    

In [12]:
# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
 
def main():
    print("Loading local data ...")
    corpus_ds, queries_ds, qrels_ds = load_local_data()
 
    corpus_rows = list(corpus_ds)
    if MAX_CORPUS_DOCS:
        corpus_rows = corpus_rows[:MAX_CORPUS_DOCS]
 
    queries_rows = list(queries_ds)
    if MAX_QUERIES:
        queries_rows = queries_rows[:MAX_QUERIES]
    query_ids = {str(r["_id"]) for r in queries_rows}
 
    relevant = build_relevant_map(qrels_ds, query_ids)
    print(f"Corpus docs: {len(corpus_rows)} | Queries: {len(queries_rows)} | Queries with qrels: {len(relevant)}")
 
    print(f"Loading embedding model {EMBED_MODEL} ...")
    model = SentenceTransformer(EMBED_MODEL)
 
    query_texts = [r["text"] for r in queries_rows]
    query_vecs = np.array(model.encode(query_texts, normalize_embeddings=True, show_progress_bar=True))
 
    results = []
    for splitter_name, configs in SWEEP_CONFIG.items():
        for chunk_size, overlap in configs:
            try:
                chunks = build_chunks(corpus_rows, splitter_name, chunk_size, overlap)
            except ImportError as e:
                print(f"Skipping splitter '{splitter_name}': missing dependency ({e})")
                continue
 
            chunk_texts = [c.text for c in chunks]
            chunk_doc_ids = [c.doc_id for c in chunks]
            chunk_vecs = np.array(model.encode(chunk_texts, normalize_embeddings=True,
                                                batch_size=64, show_progress_bar=True))
 
            for retriever_name in RETRIEVERS_TO_TRY:
                print(f"\n=== splitter={splitter_name}, chunk_size={chunk_size}, overlap={overlap}, "
                      f"retriever={retriever_name} ===")
                retriever_fn = RETRIEVER_REGISTRY[retriever_name]
                try:
                    topk_idx = retriever_fn(chunk_texts, chunk_vecs, query_texts, query_vecs, MAX_K)
                except ImportError as e:
                    print(f"  Skipping retriever '{retriever_name}': missing dependency ({e})")
                    continue
 
                per_k_recalls = {k: [] for k in K_VALUES}
                per_k_ndcgs = {k: [] for k in K_VALUES}
 
                for qi, row in enumerate(queries_rows):
                    qid = str(row["_id"])
                    rel_docs = relevant.get(qid)
                    if not rel_docs:
                        continue
                    retrieved_doc_ids = [chunk_doc_ids[i] for i in topk_idx[qi]]
                    for k in K_VALUES:
                        per_k_recalls[k].append(recall_at_k(retrieved_doc_ids, rel_docs, k))
                        per_k_ndcgs[k].append(ndcg_at_k(retrieved_doc_ids, rel_docs, k))
 
                for k in K_VALUES:
                    results.append({
                        "splitter": splitter_name,
                        "chunk_size": chunk_size,
                        "overlap": overlap,
                        "retriever": retriever_name,
                        "k": k,
                        "n_chunks": len(chunks),
                        "n_queries_scored": len(per_k_recalls[k]),
                        "recall": np.mean(per_k_recalls[k]) if per_k_recalls[k] else float("nan"),
                        "ndcg": np.mean(per_k_ndcgs[k]) if per_k_ndcgs[k] else float("nan"),
                    })
 
    df = pd.DataFrame(results).sort_values(["k", "ndcg"], ascending=[True, False])
    print("\n=== Full comparison: splitter x chunk config x retriever x k ===")
    print(df.to_string(index=False))
 
    #out_path = "finder_sweep_results.csv"
    df.to_csv(OUT_PATH_SWEEPRESULT, index=False)
    print(f"\nSaved results to {OUT_PATH_SWEEPRESULT}")
 
 
if __name__ == "__main__":
    main()

Loading local data ...
Corpus docs: 13867 | Queries: 200 | Queries with qrels: 61
Loading embedding model sentence-transformers/all-MiniLM-L6-v2 ...


Batches: 100%|██████████| 709/709 [00:45<00:00, 15.47it/s]



=== splitter=char, chunk_size=256, overlap=64, retriever=faiss_flat ===
Shape of Chunk Vec 384

=== splitter=char, chunk_size=256, overlap=64, retriever=faiss_hnsw ===

=== splitter=char, chunk_size=256, overlap=64, retriever=chroma ===

=== splitter=char, chunk_size=256, overlap=64, retriever=bm25 ===


Batches: 100%|██████████| 384/384 [00:40<00:00,  9.59it/s]



=== splitter=char, chunk_size=512, overlap=100, retriever=faiss_flat ===
Shape of Chunk Vec 384

=== splitter=char, chunk_size=512, overlap=100, retriever=faiss_hnsw ===

=== splitter=char, chunk_size=512, overlap=100, retriever=chroma ===

=== splitter=char, chunk_size=512, overlap=100, retriever=bm25 ===


...


=== splitter=recursive_char, chunk_size=512, overlap=100, retriever=faiss_flat ===
Shape of Chunk Vec 384

=== splitter=recursive_char, chunk_size=512, overlap=100, retriever=faiss_hnsw ===

=== splitter=recursive_char, chunk_size=512, overlap=100, retriever=chroma ===

=== splitter=recursive_char, chunk_size=512, overlap=100, retriever=bm25 ===


Batches: 100%|██████████| 336/336 [00:35<00:00,  9.56it/s]



=== splitter=token, chunk_size=128, overlap=20, retriever=faiss_flat ===
Shape of Chunk Vec 384

=== splitter=token, chunk_size=128, overlap=20, retriever=faiss_hnsw ===

=== splitter=token, chunk_size=128, overlap=20, retriever=chroma ===

=== splitter=token, chunk_size=128, overlap=20, retriever=bm25 ===


Batches: 100%|██████████| 244/244 [00:28<00:00,  8.67it/s]



=== splitter=token, chunk_size=256, overlap=40, retriever=faiss_flat ===
Shape of Chunk Vec 384

=== splitter=token, chunk_size=256, overlap=40, retriever=faiss_hnsw ===

=== splitter=token, chunk_size=256, overlap=40, retriever=chroma ===

=== splitter=token, chunk_size=256, overlap=40, retriever=bm25 ===

=== Full comparison: splitter x chunk config x retriever x k ===
      splitter  chunk_size  overlap  retriever  k  n_chunks  n_queries_scored   recall     ndcg
recursive_char         512      100     chroma  1     29996                61 0.171311 0.213115
         token         256       40 faiss_flat  1     15610                61 0.167213 0.213115
         token         256       40 faiss_hnsw  1     15610                61 0.167213 0.213115
          char         256       64 faiss_flat  1     45355                61 0.167213 0.196721
          char         256       64 faiss_hnsw  1     45355                61 0.167213 0.196721
          char         256       64     chroma  1